# 05 · Live demo: speak Telugu, get text

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pavanadityav-ai/telugu-asr-whisper/blob/main/notebooks/05_demo.ipynb)

Opens a small web app. Record your voice or upload an audio file, and the fine-tuned model writes it in Telugu. It also prints a public link you can open on your phone for about 72 hours.

**Runtime:** GPU recommended. Uses the model saved by notebook 03 (falls back to plain `whisper-small` if it is not there).

In [ ]:
!pip install -q gradio
import os, sys
if not os.path.isdir("telugu-asr-whisper"):
    !git clone -q https://github.com/pavanadityav-ai/telugu-asr-whisper
sys.path.insert(0, "telugu-asr-whisper")

from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/telugu-asr-whisper"  # results are saved here
os.makedirs(OUT, exist_ok=True)
print("Saving results to:", OUT)

In [ ]:
import torch
from transformers import pipeline

MODEL_PATH = f"{OUT}/whisper-small-te"
if not os.path.isdir(MODEL_PATH):
    print("Fine-tuned model not found, using the original whisper-small")
    MODEL_PATH = "openai/whisper-small"

asr = pipeline("automatic-speech-recognition", model=MODEL_PATH,
               device=0 if torch.cuda.is_available() else -1,
               chunk_length_s=30)  # long recordings are cut into 30-second pieces

def speech_to_text(audio_path):
    if audio_path is None:
        return "Please record or upload some audio."
    result = asr(audio_path, generate_kwargs={"language": "te", "task": "transcribe"})
    return result["text"]

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=speech_to_text,
    inputs=gr.Audio(sources=["microphone", "upload"], type="filepath", label="Telugu speech"),
    outputs=gr.Textbox(label="Telugu text"),
    title="Telugu speech to text",
    description="Whisper-small fine-tuned on Google FLEURS Telugu.",
)
demo.launch(share=True)